# Module 13 — Langfuse + eval RAG

Savoir si le chat **hallucine**, **refuse** trop, ou si le one-shot **casse** sur les questions difficiles.

**Prérequis** :
- Module 12 *done*
- Pour les cas `grounded` : `uv run presslake index` + `embed`
- Langfuse UI (optionnel) : `docker compose --profile langfuse up -d`

Tuto : [`docs/modules/13-langfuse.md`](../docs/modules/13-langfuse.md) · ADR [0007](../docs/adr/0007-langfuse-eval-local.md)

## Cas A — Charger le jeu YAML

Le contrat d'eval vit dans git (`config/eval/rag-v1.yml`), pas dans l'UI Langfuse.

In [1]:
from presslake.eval.dataset import load_eval_set

eval_set = load_eval_set()
print(eval_set.name, "v" + str(eval_set.version), eval_set.path)
for case in eval_set.cases:
    print(f"  {case.id:30} {case.difficulty:10} {case.expect:10} {case.question[:50]}…")

rag-v1 v1 /home/anthony-marais/Documents/data_project/config/eval/rag-v1.yml
  refuse-nonce-xylophonie        one_shot   refuse     Quelle est la population de XylophonieEval9f3a-Zor…
  refuse-nonce-licorne           one_shot   refuse     Combien de licornes QzxtmPlk-Eval vivent dans le c…
  grounded-himalaya              one_shot   grounded   Que dit le corpus sur la crue dans l'Himalaya ?…
  grounded-nepal                 one_shot   grounded   Que dit le corpus sur le Népal ?…
  hard-comparer-deux-sujets      hard       grounded   Compare deux sujets distincts du corpus (pays ou é…


## Cas B — Scorer un refus (sans index, sans LLM)

Question hors corpus → retrieve vide → `expect: refuse` doit être OK.

In [2]:
from presslake.eval.score import score_case
from presslake.rag.chat import answer_question

refuse_case = next(c for c in eval_set.cases if c.expect == "refuse")
result = answer_question(refuse_case.question, skip_llm=True)
score = score_case(refuse_case, result, skip_llm=True)
print(refuse_case.id, "→", "OK" if score.ok else "KO", score.reason)
print("passages:", len(result.passages), "refused:", result.refused)

/home/anthony-marais/Documents/data_project/src/presslake/vector/embed.py:13: UserWarning: The model sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  return TextEmbedding(model_name=embedding_model())


refuse-nonce-xylophonie → OK refus / hors corpus OK
passages: 0 refused: True


## Cas C — Tout le set en retrieve seul

Équivalent CLI : `uv run presslake eval --skip-llm`

In [3]:
from presslake.eval.run import format_report, run_eval

loaded, scores = run_eval(skip_llm=True)
print(format_report(loaded, scores))

Eval rag-v1 v1 (/home/anthony-marais/Documents/data_project/config/eval/rag-v1.yml)
5/5 OK

  [OK] refuse-nonce-xylophonie  (one_shot/refuse)  retrieve=non  refus / hors corpus OK
  [OK] refuse-nonce-licorne  (one_shot/refuse)  retrieve=non  refus / hors corpus OK
  [OK] grounded-himalaya  (one_shot/grounded)  retrieve=oui  passages trouvés (retrieve seul)
  [OK] grounded-nepal  (one_shot/grounded)  retrieve=oui  passages trouvés (retrieve seul)
  [OK] hard-comparer-deux-sujets  (hard/grounded)  retrieve=oui  passages trouvés (retrieve seul)

  one_shot : 4/4
  hard     : 1/1  (ADR 0006 — limite du one-shot)


## Cas D — one_shot vs hard (ADR 0006)

On ne passe **pas** en retrieve agentique au 13. On mesure seulement si le one-shot perd sur `hard`.

In [4]:
from collections import defaultdict

by_diff: dict[str, list] = defaultdict(list)
for s in scores:
    by_diff[s.difficulty].append(s)

for difficulty, group in by_diff.items():
    ok = sum(1 for x in group if x.ok)
    print(f"{difficulty}: {ok}/{len(group)}")
    for x in group:
        print(f"  {'OK' if x.ok else 'KO'} {x.case_id} — {x.reason}")

one_shot: 4/4
  OK refuse-nonce-xylophonie — refus / hors corpus OK
  OK refuse-nonce-licorne — refus / hors corpus OK
  OK grounded-himalaya — passages trouvés (retrieve seul)
  OK grounded-nepal — passages trouvés (retrieve seul)
hard: 1/1
  OK hard-comparer-deux-sujets — passages trouvés (retrieve seul)


## Cas E — Santé Langfuse (si profil lancé)

`docker compose --profile langfuse up -d` puis UI http://localhost:3100

In [5]:
import os

import httpx
from dotenv import load_dotenv

from presslake.eval.tracing import tracing_enabled

load_dotenv()
base = os.environ.get("LANGFUSE_BASE_URL", "http://localhost:3100").rstrip("/")
print("LANGFUSE_BASE_URL         :", base)
print("LANGFUSE_TRACING_ENABLED  :", tracing_enabled())

try:
    r = httpx.get(f"{base}/api/public/health", timeout=3.0)
    print("health", r.status_code, r.text[:200])
except httpx.HTTPError as exc:
    print("Langfuse UI absente (normal si profil langfuse off) :", exc)

LANGFUSE_BASE_URL         : http://localhost:3100
LANGFUSE_TRACING_ENABLED  : False
Langfuse UI absente (normal si profil langfuse off) : [Errno 111] Connection refused


## Cas F — Tracing off = pas de cloud

`tracing_enabled()` est false tant que le flag n'est pas `true` **et** que les trois variables (clés + URL) sont posées. Ça évite d'envoyer le corpus vers `cloud.langfuse.com`.

In [6]:
assert tracing_enabled() is False or os.environ.get("LANGFUSE_BASE_URL", "").startswith("http://localhost"), (
    "Tracing on vers un hôte non local — vérifier LANGFUSE_BASE_URL"
)
print("garde-fou OK")

garde-fou OK
